In [2]:
import pandas as pd

url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
df = pd.read_csv(url)

df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df = df.dropna()
df = df.drop(columns=["customerID"])
df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})

df["Churn"].value_counts()

Churn
0    5163
1    1869
Name: count, dtype: int64

In [3]:
X = df.drop(columns=["Churn"])
y = df["Churn"]

print(X.shape)
print(y.shape)

(7032, 19)
(7032,)


In [4]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(X_train.shape)
print(X_test.shape)

(5625, 19)
(1407, 19)


In [5]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

colunas_texto = X_train.select_dtypes(include="object").columns
colunas_numero = X_train.select_dtypes(exclude="object").columns

preparo = ColumnTransformer([
    ("texto", OneHotEncoder(handle_unknown="ignore"), colunas_texto),
    ("numero", StandardScaler(), colunas_numero),
])

modelo = Pipeline([
    ("preparo", preparo),
    ("modelo", LogisticRegression(max_iter=1000)),
])

modelo.fit(X_train, y_train)
print("Acerto na prova:", modelo.score(X_test, y_test))

Acerto na prova: 0.8038379530916845


/tmp/ipykernel_6835/2368117373.py:6: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  colunas_texto = X_train.select_dtypes(include="object").columns


In [6]:
from sklearn.metrics import classification_report

previsoes = modelo.predict(X_test)
print(classification_report(y_test, previsoes))

              precision    recall  f1-score   support

           0       0.85      0.89      0.87      1033
           1       0.65      0.57      0.61       374

    accuracy                           0.80      1407
   macro avg       0.75      0.73      0.74      1407
weighted avg       0.80      0.80      0.80      1407

